# ♟️ MicroCNN Domain Classifier Training & Benchmarking Notebook

**Target:** Train and export an ultra-compact **MicroCNN** neural classifier ($< 1.5\text{ MB}$ ONNX, sub-$2.5\text{ ms}$ CPU inference, $>99.5\%$ accuracy) for **US-3.1.2**.

### Pipeline Overview (ADR-009):
- **Tier-1 Fast Path ($<0.4\text{ ms}$)**: Multi-feature statistical screener ($H_{\text{norm}}, ZNR, AGE, LH$).
- **Tier-2 Neural Fallback ($<0.5\text{ ms}$)**: MicroCNN triggered when Tier-1 confidence is ambiguous ($0.20 \le S \le 0.80$).
- **Screen Recaptures**: Smartphone photos of monitor screens with Moiré patterns are classified as `DomainType.PHYSICAL_3D` to enforce perspective homography rectification.

In [1]:
# 1. Environment & Dependency Setup
import sys
from pathlib import Path

# Ensure project root is on sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import math
import os
import random
import time
import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from src.domain_classifier.micro_cnn import MicroCNN, build_domain_classifier_model
from src.domain_classifier.neural_classifier import NeuralDomainClassifier
from src.domain_classifier.train_micro_cnn import (
    generate_synthetic_digital_board,
    generate_synthetic_physical_photo,
    generate_synthetic_screen_recapture,
    apply_jpeg_compression,
    ChessDomainDataset,
)
from src.schemas.contracts import DomainType

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] PyTorch Version: {torch.__version__}, Device: {device}")
print(f"[Setup] ONNX Runtime Version: {ort.__version__}")

[Setup] PyTorch Version: 2.11.0+cu128, Device: cuda
[Setup] ONNX Runtime Version: 1.28.0


c:\coding\chess_ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Ingesting Real Physical & Digital Datasets (Optional Download)

If you want to download real full datasets (e.g. ChessReD 10,800 real camera photos or Roboflow Staunton), run the downloader scripts below.

In [ ]:
# Optional: Trigger automated download of physical datasets
# !uv run python scripts/download_physical_datasets.py --dataset all
# !uv run python scripts/download_digital_datasets.py --dataset all

digital_dir = project_root / "data" / "standardized" / "digital" / "images"
physical_dir = project_root / "data" / "standardized" / "physical" / "images"

real_digital_files = list(digital_dir.glob("*.*")) if digital_dir.is_dir() else []
real_physical_files = list(physical_dir.glob("*.*")) if physical_dir.is_dir() else []

print(f"Found {len(real_digital_files)} standardized digital images in {digital_dir}")
print(f"Found {len(real_physical_files)} standardized physical images in {physical_dir}")

## 3. Visualizing Training Data & Edge-Case Generators

Let's visualize the 4 key categories:
1. **Clean Digital Boards** (Chess.com, Lichess, wood/marble themes)
2. **Compressed Digital Screenshots** (JPEG compression ringing)
3. **Real & Augmented 3D Physical Photos** (camera perspective, shadows, photon noise)
4. **Monitor Screen Recaptures with Moiré** (interference fringes, bezel glare)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Digital samples (Class 0: DIGITAL_2D)
for i in range(2):
    img_d = generate_synthetic_digital_board(size=128)
    axes[0, i].imshow(cv2.cvtColor(img_d, cv2.COLOR_BGR2RGB))
    axes[0, i].set_title("Digital 2D: Clean/Themed")
    axes[0, i].axis("off")

for i in range(2, 4):
    img_d_jpeg = apply_jpeg_compression(generate_synthetic_digital_board(size=128), quality=30)
    axes[0, i].imshow(cv2.cvtColor(img_d_jpeg, cv2.COLOR_BGR2RGB))
    axes[0, i].set_title("Digital 2D: JPEG Ringing (Q=30)")
    axes[0, i].axis("off")

# Physical samples (Class 1: PHYSICAL_3D)
for i in range(2):
    img_p = generate_synthetic_physical_photo(size=128)
    axes[1, i].imshow(cv2.cvtColor(img_p, cv2.COLOR_BGR2RGB))
    axes[1, i].set_title("Physical 3D: Camera Photo")
    axes[1, i].axis("off")

for i in range(2, 4):
    img_moire = generate_synthetic_screen_recapture(size=128)
    axes[1, i].imshow(cv2.cvtColor(img_moire, cv2.COLOR_BGR2RGB))
    axes[1, i].set_title("Physical 3D: Screen Recapture (Moiré)")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

## 4. MicroCNN Model Architecture & Parameter Inspection

In [ ]:
model = MicroCNN(num_classes=2, dropout_rate=0.20).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print(f"  MicroCNN Total Trainable Parameters: {total_params:,}")
print(f"  Estimated FP32 ONNX Size: ~{total_params * 4 / (1024*1024):.2f} MB (< 1.5 MB limit)")
print("=" * 60)
print(model)

## 5. Training Pipeline (with `tqdm` Progress Tracking)

In [ ]:
# Hyperparameters
NUM_TRAIN_SAMPLES = 4000
NUM_VAL_SAMPLES = 800
BATCH_SIZE = 64
EPOCHS = 25
LEARNING_RATE = 1e-3

# Create Datasets
train_dataset = ChessDomainDataset(
    num_samples=NUM_TRAIN_SAMPLES,
    real_digital_dir=digital_dir,
    real_physical_dir=physical_dir,
    is_train=True,
)
val_dataset = ChessDomainDataset(
    num_samples=NUM_VAL_SAMPLES,
    real_digital_dir=digital_dir,
    real_physical_dir=physical_dir,
    is_train=False,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0
start_time = time.time()

epoch_pbar = tqdm(range(1, EPOCHS + 1), desc="Training Progress", unit="epoch")

for epoch in epoch_pbar:
    model.train()
    train_loss, train_correct, total_train = 0.0, 0, 0

    train_batch_pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Train]",
        leave=False,
        unit="batch",
    )
    for images, labels in train_batch_pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_loss = loss.item()
        train_loss += batch_loss * images.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        total_train += images.size(0)

        train_batch_pbar.set_postfix({
            "loss": f"{batch_loss:.4f}",
            "acc": f"{(train_correct / total_train):.2%}",
        })

    scheduler.step()

    # Validation
    model.eval()
    val_loss, val_correct, total_val = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            total_val += images.size(0)

    t_acc = train_correct / total_train
    v_acc = val_correct / total_val
    t_loss = train_loss / total_train
    v_loss = val_loss / total_val

    history["train_loss"].append(t_loss)
    history["val_loss"].append(v_loss)
    history["train_acc"].append(t_acc)
    history["val_acc"].append(v_acc)

    if v_acc > best_val_acc:
        best_val_acc = v_acc

    epoch_pbar.set_postfix({
        "Train Loss": f"{t_loss:.4f}",
        "Val Acc": f"{v_acc:.2%}",
        "Best Val": f"{best_val_acc:.2%}",
    })

print(f"\nTraining Complete in {time.time() - start_time:.1f}s. Best Val Accuracy: {best_val_acc:.2%}")

## 6. Training & Validation Curves

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS + 1), history["train_loss"], label="Train Loss", color="tab:blue", marker="o")
plt.plot(range(1, EPOCHS + 1), history["val_loss"], label="Val Loss", color="tab:orange", marker="s")
plt.title("Cross-Entropy Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS + 1), [a * 100 for a in history["train_acc"]], label="Train Accuracy", color="tab:green", marker="o")
plt.plot(range(1, EPOCHS + 1), [a * 100 for a in history["val_acc"]], label="Val Accuracy", color="tab:red", marker="s")
plt.title("Domain Classification Accuracy (%)")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.tight_layout()
plt.show()

## 7. Exporting to ONNX Format

In [ ]:
weights_dir = project_root / "src" / "domain_classifier" / "weights"
weights_dir.mkdir(parents=True, exist_ok=True)
onnx_path = weights_dir / "domain_classifier_microcnn.onnx"

model.eval().cpu()
dummy_input = torch.randn(1, 3, 128, 128, dtype=torch.float32)

torch.onnx.export(
    model,
    dummy_input,
    str(onnx_path),
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
)

file_size_mb = onnx_path.stat().st_size / (1024 * 1024)
print(f"[Export] Saved ONNX Model: {onnx_path}")
print(f"[Export] Model File Size: {file_size_mb:.3f} MB (Constraint: < 1.5 MB)")
assert file_size_mb < 1.5, "Model size exceeds 1.5 MB limit!"

## 8. ONNX Runtime CPU Latency Benchmark (with `tqdm`)

In [ ]:
classifier = NeuralDomainClassifier(model_path=onnx_path, device="cpu")
test_board = generate_synthetic_digital_board(size=128)

# Warmup
for _ in range(10):
    classifier.classify(test_board)

latencies = []
for _ in tqdm(range(100), desc="Benchmarking ONNX CPU Latency", unit="inf"):
    t0 = time.perf_counter()
    res = classifier.classify(test_board)
    latencies.append((time.perf_counter() - t0) * 1000.0)

mean_lat = np.mean(latencies)
p50_lat = np.percentile(latencies, 50)
p95_lat = np.percentile(latencies, 95)
p99_lat = np.percentile(latencies, 99)

print("=" * 55)
print("  ⚡ ONNX RUNTIME CPU LATENCY BENCHMARK (Batch Size = 1)")
print("=" * 55)
print(f"  Mean Latency: {mean_lat:.4f} ms (Target: < 2.5 ms)")
print(f"  P50  Latency: {p50_lat:.4f} ms")
print(f"  P95  Latency: {p95_lat:.4f} ms")
print(f"  P99  Latency: {p99_lat:.4f} ms")
print("=" * 55)
assert mean_lat < 2.5, f"Latency {mean_lat:.2f} ms exceeds 2.5 ms limit!"

## 9. Validation on Edge Cases (Recaptured Displays & Moiré)

In [ ]:
print("Testing Screen Recapture with Moiré Classification:")
correct_recaptures = 0
total_recaptures = 100

for _ in tqdm(range(total_recaptures), desc="Evaluating Screen Recaptures", unit="sample"):
    recapture_img = generate_synthetic_screen_recapture(size=128)
    result = classifier.classify(recapture_img)
    if result.domain == DomainType.PHYSICAL_3D:
        correct_recaptures += 1

recapture_acc = (correct_recaptures / total_recaptures) * 100
print(f"Screen Recapture Routing to Domain.PHYSICAL_3D: {correct_recaptures}/{total_recaptures} ({recapture_acc:.1f}%)")
assert recapture_acc >= 98.0, "Screen recaptures must route to PHYSICAL_3D for homography rectification!"